In [3]:
import cv2
import numpy as np
import os
import random

In [1]:
#Use Ghostscript to convert it to images
import subprocess

cmd = [
    "gswin64c",
    "-sDEVICE=png16m",
    "-r300",
    "-sOutputFile=C:\\Users\\user\\OCR\\images\\page_%03d.png",
    "C:\\Users\\user\\OCR\\sample.pdf",
    "-dBATCH",
    "-dNOPAUSE"
]

subprocess.run(cmd, check=True)


CompletedProcess(args=['gswin64c', '-sDEVICE=png16m', '-r300', '-sOutputFile=C:\\Users\\user\\OCR\\images\\page_%03d.png', 'C:\\Users\\user\\OCR\\sample.pdf', '-dBATCH', '-dNOPAUSE'], returncode=0)

In [6]:
# ===============================
# CONFIG
# ===============================
INPUT_IMAGE = "C:\\Users\\user\\OCR\\images\\page_002.PNG"   # place your image here
OUTPUT_DIR = "synthetic_data/augmented2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

img = cv2.imread(INPUT_IMAGE)
print(img)
h, w = img.shape[:2]

def save(img, folder, name):
    path = os.path.join(OUTPUT_DIR, folder)
    os.makedirs(path, exist_ok=True)
    cv2.imwrite(os.path.join(path, name), img)

[[[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 ...

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]
  ...
  [255 255 255]
  [255 255 255]
  [255 255 255]]]


In [7]:
# ===============================
# 1. ROTATION & SKEW (6 images)
# ===============================
angles = [-7, -5, -3, 3, 5, 7]
for a in angles:
    M = cv2.getRotationMatrix2D((w//2, h//2), a, 1)
    rot = cv2.warpAffine(img, M, (w, h), borderValue=(255,255,255))
    save(rot, "rotate", f"rotate_{a}.png")

# ===============================
# 2. BRIGHTNESS / CONTRAST (6 images)
# ===============================
params = [(0.7,-40),(0.85,-20),(1.2,20),(1.4,40),(1.6,60),(0.9,10)]
for i,(a,b) in enumerate(params):
    bc = cv2.convertScaleAbs(img, alpha=a, beta=b)
    save(bc, "brightness", f"bc_{i}.png")

# ===============================
# 3. BLUR (4 images)
# ===============================
kernels = [(3,3),(5,5),(7,7),(9,9)]
for k in kernels:
    blur = cv2.GaussianBlur(img, k, 0)
    save(blur, "blur", f"blur_{k[0]}.png")

# ===============================
# 4. NOISE (4 images)
# ===============================
for i in range(4):
    noise = np.random.normal(0, random.randint(10,25), img.shape)
    noisy = np.clip(img + noise, 0, 255).astype(np.uint8)
    save(noisy, "noise", f"noise_{i}.png")

# ===============================
# 5. PERSPECTIVE WARP (4 images)
# ===============================
for i in range(4):
    shift = random.randint(20,40)
    src = np.float32([[0,0],[w,0],[0,h],[w,h]])
    dst = np.float32([
        [shift,shift],
        [w-shift,shift],
        [shift,h-shift],
        [w-shift,h-shift]
    ])
    M = cv2.getPerspectiveTransform(src, dst)
    warp = cv2.warpPerspective(img, M, (w,h), borderValue=(255,255,255))
    save(warp, "perspective", f"warp_{i}.png")

# ===============================
# 6. LOW RES / COMPRESSION (3 images)
# ===============================
scales = [0.5, 0.6, 0.7]
for s in scales:
    small = cv2.resize(img, None, fx=s, fy=s)
    low = cv2.resize(small, (w,h))
    save(low, "lowres", f"lowres_{s}.png")

# ===============================
# 7. COMBINED REALISTIC EFFECTS (8 images)
# ===============================
for i in range(8):
    angle = random.randint(-5,5)
    alpha = random.uniform(0.8,1.4)
    beta = random.randint(-30,30)
    
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
    out = cv2.warpAffine(img, M, (w,h), borderValue=(255,255,255))
    out = cv2.convertScaleAbs(out, alpha=alpha, beta=beta)
    out = cv2.GaussianBlur(out, (5,5), 0)
    
    save(out, "combined", f"combo_{i}.png")

print("✅ Synthetic image generation completed (~35 images).")


✅ Synthetic image generation completed (~35 images).
